In [2]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
# import matplotlib.pyplot as plt; plt.style.use('seaborn')
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

filename = 'normal.xlsx'
sheetname = 'age56'
df = pd.read_excel(filename, sheetname, header=0)

X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']]

y = df['fc (MPa)']

# X= dataset.iloc[:, 1:]
# y = dataset.iloc[:, 0]
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)
Xtrain_column_name=list(Xtrain.columns)

n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
max_features = ['auto', 'sqrt']
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
bootstrap = [True, False]
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

rf = RandomForestRegressor()
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid,
                               n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = 12)
rf_random.fit(Xtrain, ytrain)
rf_random.best_params_

rf_model = rf_random.best_estimator_

# Predict test set data
random_forest_predict=rf_model.predict(Xtest)

# Verify the accuracy
random_forest_R2=metrics.r2_score(ytest,random_forest_predict)
random_forest_RMSE=metrics.mean_squared_error(ytest,random_forest_predict)**0.5
random_forest_MAE=metrics.mean_absolute_error(ytest,random_forest_predict)
print('R-squared is {0}, RMSE is {1}, and MAE is {2}.'.format(random_forest_R2,
                                                              random_forest_RMSE,
                                                              random_forest_MAE))


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Fitting 3 folds for each of 100 candidates, totalling 300 fits


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
123 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
80 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/base.py", line 1466, in wrapper
    estimator._validate_params()
  File "/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/sklearn/base.py", lin

R-squared is 0.6203486411058188, RMSE is 10.446527830333357, and MAE is 7.4069751981184595.


In [3]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
# import matplotlib.pyplot as plt; plt.style.use('seaborn')
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
# from bayes_opt import BayesianOptimization
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

# Load the dataset
filename = 'normal.xlsx'
sheetname = 'age56'
df = pd.read_excel(filename, sheetname, header=0)

# Define features and target
X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']]
y = df['fc (MPa)']

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_column_name = list(X_train.columns)

# Parameter grid for randomized search
param_grid = {
    'max_depth': np.arange(3, 10, 1),
    'colsample_bytree': np.arange(0.5, 1.0, 0.1),
    'gamma': np.arange(0, 0.5, 0.1),
    'learning_rate': np.arange(0.01, 0.1, 0.01),
    'n_estimators': [100, 200, 300, 400, 500]
}

# Initialize the XGBRegressor
xgb = XGBRegressor(objective='reg:squarederror')

# RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(xgb, param_distributions=param_grid, n_iter=50, scoring='neg_mean_squared_error', cv=3, verbose=3, random_state=42, n_jobs=24)

# Fit the model
random_search.fit(X_train, y_train)

# Get the best model
best_xgb = random_search.best_estimator_

# Make predictions on the test data
predictions = best_xgb.predict(X_test)

# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, predictions)

# Calculate R² score for both train and test sets
train_r2 = r2_score(y_train, best_xgb.predict(X_train))
test_r2 = r2_score(y_test, predictions)

# Print the results
print("Best estimator: ", best_xgb)
print("Best parameters: ", random_search.best_params_)
print("Best validation score: ", random_search.best_score_)
print("MSE on test data: ", mse)
print("R² on training data: ", train_r2)
print("R² on test data: ", test_r2)


Fitting 3 folds for each of 50 candidates, totalling 150 fits


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best estimator:  XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.0, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.04, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)
Best parameters:  {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.04, 'gamma': 0.0, 'colsample_bytree': 0.6}
Best validation score:  -98.56226700439926
MSE on test data:  115.77784694370546
R² on training data:  0.8222855098642331
R²

In [4]:
import optuna
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


# 导入数据
df = pd.read_excel('normal.xlsx', sheet_name='age56')
# 删除缺失值
df.dropna(inplace=True)

X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
Y = df['fc (MPa)'].values
# 提取数据
# Y = df.iloc[:, 0].values
# X = df.iloc[:, 1:].values

# 数据标准化
x_mean = X.mean(0)
x_std = X.std(0)
X_normal = (X - x_mean) / x_std

y_mean = Y.mean()
y_std = Y.std()
Y_normal = (Y - y_mean) / y_std
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X_normal, Y_normal, train_size=0.80, random_state=42)



def create_model(trial):
    # 为超参数定义搜索空间
    layers = trial.suggest_int('layers', 1, 5)
    neurons = trial.suggest_int('neurons', 16, 256)
    learn_rate = trial.suggest_float('learn_rate', 1e-4, 1e-1,log=True)

    model = Sequential()
    model.add(Dense(neurons, input_dim=X.shape[1], activation='relu', kernel_initializer='he_normal'))
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu', kernel_initializer='he_normal'))
    model.add(Dense(1, activation='linear'))

    optimizer = tf.keras.optimizers.Adam(learn_rate)
    model.compile(loss='mean_squared_error', optimizer=optimizer)
    return model


def objective(trial):
    batch_size = trial.suggest_int('batch_size', 16, 64)

    model = create_model(trial)
    model.fit(X_train, y_train, epochs=100, batch_size=batch_size, verbose=0, validation_split=0.1)

    y_pred = model.predict(X_test)
    return mean_squared_error(y_test, y_pred)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

best_model = create_model(study.best_trial)
batch_size = study.best_trial.params['batch_size']
best_model.fit(X_train, y_train, epochs=100, batch_size=batch_size, verbose=0)

# 保存最优模型
best_model.save('best_model.h5')

# 计算 R2 值
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
print('Train R2:', train_r2)
print('Test R2:', test_r2)

2024-10-25 17:04:08.228613: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[I 2024-10-25 17:04:29,868] A new study created in memory with name: no-name-3969e039-72cf-4358-995b-f57e3ea464d4
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


[I 2024-10-25 17:04:37,904] Trial 0 finished with value: 0.4921875641805749 and parameters: {'batch_size': 48, 'layers': 4, 'neurons': 92, 'learn_rate': 0.00044592672068011994}. Best is trial 0 with value: 0.4921875641805749.
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


[I 2024-10-25 17:04:46,064] Trial 1 finished with value: 0.5263113504587481 and parameters: {'batch_size': 29, 'layers': 2, 'neurons': 142, 'learn_rate': 0.0042186731682218965}. Best is trial 0 with value: 0.4921875641805749.
/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


[I 2024-10-25 21:50:25,709] Trial 99 finished with value: 0.48922319626545346 and parameters: {'batch_size': 22, 'layers': 3, 'neurons': 170, 'learn_rate': 0.019332733425766555}. Best is trial 54 with value: 0.40576160318809706.


Number of finished trials: 100
Best trial: {'batch_size': 23, 'layers': 3, 'neurons': 148, 'learn_rate': 0.004179170906619083}


/Users/zhangtianjie/opt/anaconda3/envs/ag/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Train R2: 0.8634555319802822
Test R2: 0.5393289064444557


## emsemble ANN

In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score

# Custom PyTorch model
class ANNModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(ANNModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Load and preprocess the data
# df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
df = pd.read_excel('normal.xlsx', sheet_name='age56')
df.dropna(inplace=True)

X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values
Y = df['fc (MPa)'].values

# Normalize the data
x_mean = X.mean(0)
x_std = X.std(0)
X_normal = (X - x_mean) / x_std

y_mean = Y.mean()
y_std = Y.std()
Y_normal = (Y - y_mean) / y_std

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=42)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Hyperparameters
# params = {'batch_size': 23, 'layers': 3, 'neurons': 197, 'learn_rate': 0.003289282795719042}
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0075788652034274205}

# Define model, loss function, and optimizer
def build_model(input_dim, layers, neurons):
    model = ANNModel(input_dim=input_dim, layers=layers, neurons=neurons)
    return model

# K-Fold Cross-validation setup
kf = KFold(n_splits=100, shuffle=True, random_state=42)
models = []
preds_train = np.zeros_like(y_train)

# Loss function and optimizer
loss_fn = nn.MSELoss()

# Perform K-fold training
for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    X_val_fold = X_train_tensor[val_index]
    y_val_fold = y_train_tensor[val_index]
    
    model = build_model(input_dim=X_train.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Training loop
    for epoch in range(300):  # You can increase the number of epochs if necessary
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(X_train_fold)
        loss = loss_fn(y_pred_train, y_train_fold)
        loss.backward()
        optimizer.step()

    # Save the model
    models.append(model)
    
    # Generate validation predictions
    model.eval()
    with torch.no_grad():
        preds_train[val_index] = model(X_val_fold).numpy().flatten()

# Evaluate cross-validation score on the train set
cv_score = r2_score(y_train, preds_train)
print(f'Cross-validation R2 score: {cv_score}')

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test)

for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions
preds_test /= len(models)

# Calculate R2 score on the test data
test_score = r2_score(y_test, preds_test)
print(f'Test R2 score: {test_score}')

Cross-validation R2 score: 0.5475734566113051
Test R2 score: 0.5216301031720048


## KINN

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score

# Custom ANN Model with custom loss
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Custom loss function using the fitted parameters
def custom_loss(outputs, targets, inputs, a, b):
    mse_loss = nn.MSELoss()(outputs, targets)
    
    # Extract features needed for the fitted equation
    # AGE = inputs[:, 0]  # Assuming AGE is the first feature
    wb = inputs[:, -4]  # Assuming wb is the seventh feature from the end
    
    # Clamp AGE to avoid log of zero or negative numbers
    # AGE = torch.clamp(AGE, min=1e-6)
    
    # Compute the fitted equation: fc = (a * log(AGE) + b) * (e * AGE^d)^(-wb)
    # fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)

    fc_pred = a * b ** (-wb)
    
    # Calculate the residual between ANN predicted outputs and fitted_fc
    residual = torch.abs(outputs - fc_pred.unsqueeze(1))
    
    # Replace any NaNs in the residual with 0.0
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e4, neginf=-1e4)
    
    # Normalize residual by comparing its mean square with the MSE
    mean_square_residual = torch.mean(residual ** 2)
    if mean_square_residual.item() > 0:  # Avoid division by zero
        residual_normalized = residual * torch.sqrt(mse_loss / mean_square_residual)
    else:
        residual_normalized = residual  # In case the residual is exactly zero
    
    # Combine MSE loss and the normalized residual
    total_loss = 0.5 * mse_loss + 0.5 * torch.mean(residual_normalized)
    
    return total_loss

# Training function
def train_model(model, optimizer, Xtrain, ytrain, epochs=300, batch_size=24, a=None, b=None):
    dataset = torch.utils.data.TensorDataset(Xtrain, ytrain)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = custom_loss(outputs, targets, inputs, a, b)
            loss.backward()
            optimizer.step()

# Load and preprocess the data
# df = pd.read_excel(r'updated_fc_predictions.xlsx', sheet_name='Sheet1')
# df.dropna(inplace=True)

df = pd.read_excel('normal.xlsx', sheet_name='age7')
df.dropna(inplace=True)

X = df[['PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%']].values

Y = df['fc (MPa)'].values

# Normalize the data
x_mean = X.mean(0)
x_std = X.std(0)
X_normal = (X - x_mean) / x_std

y_mean = Y.mean()
y_std = Y.std()
Y_normal = (Y - y_mean) / y_std

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=42)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the fitted parameters
# a = 40.50
# b = 15.29
# a = 151.01590678582858
# b = 48.9602723045581
a = 215.2799032986731
b = 50.2741077251154
# e = 6.49
# d = 0.36

# Hyperparameters
params = {'batch_size': 24, 'layers': 3, 'neurons': 232, 'learn_rate': 0.0076}

# Define model, loss function, and optimizer
def build_model(input_dim, layers, neurons):
    model = RegressionModel(input_dim=input_dim, layers=layers, neurons=neurons)
    return model

# K-Fold Cross-validation setup
kf = KFold(n_splits=20, shuffle=True, random_state=42)
models = []
preds_train = np.zeros_like(y_train)

# Perform K-fold training
for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
    X_train_fold = X_train_tensor[train_index]
    y_train_fold = y_train_tensor[train_index]
    X_val_fold = X_train_tensor[val_index]
    y_val_fold = y_train_tensor[val_index]
    
    model = build_model(input_dim=X_train.shape[1], layers=params['layers'], neurons=params['neurons'])
    optimizer = optim.Adam(model.parameters(), lr=params['learn_rate'])

    # Train the model
    train_model(model, optimizer, X_train_fold, y_train_fold, epochs=300, batch_size=params['batch_size'], a=a, b=b)

    # Save the model
    models.append(model)
    
    # Generate validation predictions
    model.eval()
    with torch.no_grad():
        preds_train[val_index] = model(X_val_fold).numpy().flatten()

# Evaluate cross-validation score on the train set
cv_score = r2_score(y_train, preds_train)
print(f'Cross-validation R2 score: {cv_score}')

# Ensemble predictions on test data
preds_test = np.zeros_like(y_test)

for model in models:
    model.eval()
    with torch.no_grad():
        preds_test += model(X_test_tensor).numpy().flatten()

# Average predictions
preds_test /= len(models)

# Calculate R2 score on the test data
test_score = r2_score(y_test, preds_test)
print(f'Test R2 score: {test_score}')


Cross-validation R2 score: -0.1924265666641889
Test R2 score: 0.48910982376113565
